<a href="https://colab.research.google.com/github/dmecoyfin/dmeyf2026/blob/ensembles_2026/ensembles/z494_TareaHogar_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Competencia 01 — Optimización Bayesiana
**Ejecución:** de inicio a fin (Run All). Única entrada: el dataset crudo `competencia_01_crudo.csv` (o `.csv.gz` / `.zip`) en `~/buckets/b1/datasets/` (la carpeta tiene que existir). Todo lo demás se lee si existe y se crea si no; si se corta, retoma desde los checkpoints.

**Feature Engineering:**
1. `ccajas_depositos` en NA para 202105.
2. `lag1`, `delta1`, `lag2`, `delta2` de todas las variables originales (escala original).
3. Rank por mes con cero fijo de TODAS las variables (originales, lags y deltas), salvo `numero_de_cliente`, `foto_mes` y `clase_ternaria`
<br>Archivo: `competencia_01_fe_naccajas_lag1_delta1_lag2_delta2_rankfinal.fst`.

**BO** — definición de z495 (Gustavo Pasquini) sin undersampling
* Rangos: `num_iterations` 64–4096, `num_leaves` 16–2048, `min_sum_hessian_in_leaf` 1e-06–0.1. Fijos: `learning_rate` 0.005, `feature_fraction` 0.5, `min_data_in_leaf` 0, `max_bin` 31.
* k-fold 5 estratificado (`lgb.cv`, `nfold = 5`, `stratified = TRUE`) sobre 202103–05, **métrica AUC**, **sin undersampling**.
* 12 puntos iniciales + **10 iteraciones bayesianas** (22 evaluaciones).

### 01 Configuración

#### 1.1 - Entorno de trabajo

In [3]:
# definir el path de trabajo La corrí en una VM de Google Cloud con 8 vCPU y 64 GB de RAM
options(repos = c(CRAN = "https://cloud.r-project.org"))   # para que install.packages no pida mirror
carpeta_base <- path.expand("~/buckets/b1")
# datasets/ TIENE que existir: de ahi se lee el dataset crudo (unica entrada). Todo lo demas se crea si falta.
if (!dir.exists(file.path(carpeta_base, "datasets")))
  stop("No existe la carpeta ", file.path(carpeta_base, "datasets"), ": crearla y copiar ahi el dataset crudo")
dir.create(file.path(carpeta_base, "exp"), showWarnings = FALSE, recursive = TRUE)
setwd(carpeta_base)

#### 1.2 - Archivos y flags de preprocesamiento
Las secciones 02 y 03 solo se ejecutan si el archivo correspondiente no existe (o si se fuerza la regeneración).
<br>El nombre del archivo con FE se arma a partir de los flags, así cada combinación queda en su propio archivo. Se guarda en `.fst`: se lee en segundos y solo los meses que se usan.

In [4]:
# fst: formato binario, lectura rapida y por rango de filas (no descomprime un .csv.gz de varios GB)
if (!require("fst")) install.packages("fst")
require("fst")
if (!require("data.table")) install.packages("data.table")
require("data.table")
if (!require("R.utils")) install.packages("R.utils")   # fread / fwrite de .csv.gz
require("R.utils")

CFG <- list()
CFG$carpeta_datasets <- file.path(carpeta_base, "datasets")
# dataset crudo: UNICO archivo de entrada. Se acepta .csv, .csv.gz o .zip (el primero que exista)
crudos <- file.path(CFG$carpeta_datasets, paste0("competencia_01_crudo", c(".csv", ".csv.gz", ".zip")))
CFG$archivo_crudo <- if (any(file.exists(crudos))) crudos[file.exists(crudos)][1] else crudos[1]
rm(crudos)
CFG$carpeta_exp <- file.path(carpeta_base, "exp")
dir.create(CFG$carpeta_exp, showWarnings = FALSE, recursive = TRUE)
CFG$archivo_clase <- file.path(CFG$carpeta_datasets, "competencia_01.csv.gz")

# flags de feature engineering: TRUE = se aplica el paso
CFG$fe <- list(
  na_ccajas_202105 = TRUE,   # 1) ccajas_depositos en NA para 202105 (antes de calcular lags)
  lag1_delta1      = TRUE,   # 2) lag1 y delta1 de las variables originales
  lag2_delta2      = TRUE,   #    lag2 y delta2 de las variables originales
  rank_final       = TRUE    # 3) DESPUES de los lags: rank por mes con cero fijo de TODAS las variables
                             #    (originales, lags y deltas) salvo numero_de_cliente, foto_mes y clase_ternaria
)

# forzar regeneracion (por ejemplo si se modifico el codigo del FE sin cambiar los flags)
CFG$regenerar_clase <- FALSE
CFG$regenerar_fe    <- FALSE

# nombre del archivo con FE segun los flags activos
sufijos <- c(na_ccajas_202105 = "naccajas", lag1_delta1 = "lag1_delta1", lag2_delta2 = "lag2_delta2", rank_final = "rankfinal")
activos <- names(CFG$fe)[unlist(CFG$fe)]
CFG$archivo_fe <- file.path(CFG$carpeta_datasets, paste0(
  "competencia_01_fe_", if (length(activos)) paste(sufijos[activos], collapse = "_") else "sin_fe", ".fst"))

# si el archivo con FE existe, verifico que sus columnas coincidan con los flags (lee solo los metadatos del .fst)
fe_valido <- function(archivo, flags) {
  if (!file.exists(archivo)) return(FALSE)
  cols <- metadata_fst(archivo)$columnNames
  tiene <- c(lag1_delta1 = any(grepl("_delta1$", cols)), lag2_delta2 = any(grepl("_delta2$", cols)),
             rank_monetarias = any(grepl("_rank$", cols)))
  esperado <- c(lag1_delta1 = isTRUE(flags$lag1_delta1), lag2_delta2 = isTRUE(flags$lag2_delta2),
                # con rank_reemplaza no hay columnas _rank (el rank queda con el nombre original)
                rank_monetarias = isTRUE(flags$rank_monetarias) && !isTRUE(flags$rank_reemplaza))
  # el NA de ccajas_depositos y el rank que reemplaza no se ven en las columnas: quedan identificados por el nombre del archivo
  ok <- identical(unname(tiene), unname(esperado))
  if (!ok) warning("Las columnas de ", basename(archivo), " no coinciden con los flags: se regenera")
  ok
}

# que hay que hacer en esta corrida
CFG$hacer_fe    <- CFG$regenerar_fe || CFG$regenerar_clase || !fe_valido(CFG$archivo_fe, CFG$fe)
CFG$hacer_clase <- CFG$regenerar_clase || (CFG$hacer_fe && !file.exists(CFG$archivo_clase))
if (CFG$hacer_clase && !file.exists(CFG$archivo_crudo))
  stop("Falta el dataset crudo: copiar competencia_01_crudo.csv (o .csv.gz / .zip) en ", CFG$carpeta_datasets)
cat("Dataset crudo:", CFG$archivo_crudo, "| existe:", file.exists(CFG$archivo_crudo), "\n")

rm(sufijos, activos, fe_valido)

cat("Generar clase_ternaria:", CFG$hacer_clase, "\n")
cat("Generar feature engineering:", CFG$hacer_fe, "\n")
cat("Archivo con FE:", CFG$archivo_fe, "\n")

Loading required package: fst

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOpt

Dataset crudo: /home/ds/buckets/b1/datasets/competencia_01_crudo.zip | existe: TRUE 
Generar clase_ternaria: FALSE 
Generar feature engineering: TRUE 
Archivo con FE: /home/ds/buckets/b1/datasets/competencia_01_fe_naccajas_lag1_delta1_lag2_delta2_rankfinal.fst 


### 02 Generación de la clase_ternaria

In [5]:
require( "data.table" )

if (CFG$hacer_clase) {

  # leo el dataset crudo
  # descomprimo el .zip en el disco local de la VM (no en el bucket) y lo leo
  if (grepl("\\.zip$", CFG$archivo_crudo)) {
    # .zip: se descomprime al disco local de la VM (no en el bucket)
    csv_local <- utils::unzip(CFG$archivo_crudo, exdir = tempdir())
    dataset <- fread(csv_local[grepl("\\.csv$", csv_local)][1])
    unlink(csv_local)
  } else {
    dataset <- fread(CFG$archivo_crudo)   # .csv o .csv.gz
  }
  cat("Dataset crudo leido:", CFG$archivo_crudo, "|", nrow(dataset), "filas x", ncol(dataset), "columnas\n")

  # calculo el periodo0 consecutivo
  dsimple <- dataset[, list(
      "pos" = .I,
      numero_de_cliente,
      periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]

  # ordeno
  setorder( dsimple, numero_de_cliente, periodo0 )

  # calculo topes
  periodo_ultimo <- dsimple[, max(periodo0) ]
  periodo_anteultimo <- periodo_ultimo - 1

  # calculo los leads de orden 1 y 2
  dsimple[, c("periodo1", "periodo2") :=
      shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

  # assign most common class values = "CONTINUA"
  dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

  # calculo BAJA+1
  dsimple[ periodo0 < periodo_ultimo &
      ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
      clase_ternaria := "BAJA+1" ]

  # calculo BAJA+2
  dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
      & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
      clase_ternaria := "BAJA+2" ]

  # pego el resultado en el dataset original y grabo
  setorder( dsimple, pos )
  dataset[, clase_ternaria := dsimple$clase_ternaria ]

  fwrite( dataset, file = CFG$archivo_clase, sep = "," )
  cat("clase_ternaria generada y guardada en:", CFG$archivo_clase, "\n")

  # libero memoria
  rm(dsimple, periodo_ultimo, periodo_anteultimo)
  gc(full = TRUE, verbose = FALSE)

} else if (CFG$hacer_fe) {

  # la clase ya existe: la leo solo porque hay que rehacer el FE
  dataset <- fread(CFG$archivo_clase)
  cat("clase_ternaria leida desde:", CFG$archivo_clase, "\n")

} else {
  cat("Se omite la seccion 02: ya existe el dataset con FE\n", CFG$archivo_fe, "\n")
}

clase_ternaria leida desde: /home/ds/buckets/b1/datasets/competencia_01.csv.gz 


In [6]:
# control de la clase (keyby no reordena el dataset)
if (exists("dataset")) dataset[, .N, keyby = list(foto_mes, clase_ternaria)]

foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202103,BAJA+1,1019
202103,BAJA+2,960
202103,CONTINUA,160921
202104,BAJA+1,964
202104,BAJA+2,1139
202104,CONTINUA,161181
202105,BAJA+1,1143
202105,BAJA+2,870
202105,CONTINUA,161755


### 03 Feature Engineering

In [7]:
if (CFG$hacer_fe) {
  dataset_fe <- dataset
  rm(dataset)
  gc(full = TRUE, verbose = FALSE)
  cat(nrow(dataset_fe), "filas x", ncol(dataset_fe), "columnas\n")
} else {
  cat("Se omite la seccion 03: se usara", CFG$archivo_fe, "\n")
}

983061 filas x 155 columnas


In [8]:
mostrar_fe <- function(dt, nuevas, titulo, ver = head(nuevas, 10L), n = 5L, filas = NULL) {
  cat("\n=====", titulo, "=====\n")
  cat("Dimensiones:", format(nrow(dt), big.mark = ".", decimal.mark = ","), "filas x", ncol(dt),
      "columnas |", round(as.numeric(object.size(dt)) / 1024^3, 2), "GB\n")
  cat("Columnas nuevas/modificadas en este paso:", length(nuevas), "\n")
  idx <- if (is.null(filas)) seq_len(min(n, nrow(dt))) else head(which(filas), n)
  print(dt[idx, unique(c("numero_de_cliente", "foto_mes", ver)), with = FALSE])
  if (length(ver) < length(nuevas)) cat("(se muestran", length(ver), "de", length(nuevas), "columnas)\n")
  invisible(NULL)
}

#### 03.0 ccajas_depositos en NA para 202105
Corrección de los datos de base: se pone en NA **antes** de calcular los lags

In [9]:
if (CFG$hacer_fe && isTRUE(CFG$fe$na_ccajas_202105)) {
  stopifnot("ccajas_depositos" %in% names(dataset_fe))
  cat("ccajas_depositos en 202105 antes:\n")
  print(summary(dataset_fe[foto_mes == 202105, ccajas_depositos]))

  # data.table convierte NA al tipo de la columna (integer o numeric)
  dataset_fe[foto_mes == 202105, ccajas_depositos := NA]

  cat("Registros de 202105 con ccajas_depositos NA:", dataset_fe[foto_mes == 202105, sum(is.na(ccajas_depositos))],
      "de", dataset_fe[foto_mes == 202105, .N], "\n")
} else cat("NA ccajas_depositos 202105: se omite\n")

ccajas_depositos en 202105 antes:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
      0       0       0       0       0       0 
Registros de 202105 con ccajas_depositos NA: 163768 de 163768 


#### 03.1 Lag1 y Delta1

In [10]:
if (CFG$hacer_fe && isTRUE(CFG$fe$lag1_delta1)) {

  # ---- Feature Engineering: lag1 y delta1 de todas las variables numericas originales ----
  setorder(dataset_fe, numero_de_cliente, foto_mes)

  cols_excluir <- c("numero_de_cliente", "foto_mes", "clase_ternaria")
  cols_fe <- setdiff(names(dataset_fe)[sapply(dataset_fe, is.numeric)], cols_excluir)
  cols_fe <- cols_fe[!grepl("_rank$", cols_fe)]   # en modo "agrega", las columnas _rank no se lagean

  # periodo consecutivo para detectar meses faltantes
  dataset_fe[, periodo0     := as.integer(foto_mes / 100) * 12L + foto_mes %% 100L]
  dataset_fe[, periodo_prev := shift(periodo0, 1L, type = "lag")]
  dataset_fe[, cliente_prev := shift(numero_de_cliente, 1L, type = "lag")]

  ok <- dataset_fe[, numero_de_cliente == cliente_prev & (periodo0 - periodo_prev) == 1L]
  ok[is.na(ok)] <- FALSE
  idx_sin_lag <- which(!ok)

  for (col in cols_fe) {
    v <- shift(dataset_fe[[col]], 1L, type = "lag")
    v[idx_sin_lag] <- NA
    set(dataset_fe, j = paste0(col, "_lag1"),   value = v)
    set(dataset_fe, j = paste0(col, "_delta1"), value = dataset_fe[[col]] - v)
  }

  dataset_fe[, c("periodo0", "periodo_prev", "cliente_prev") := NULL]
  cat("Columnas numéricas a aplicar lag1 y delta1:", length(cols_fe), "| Columnas totales del dataset:", ncol(dataset_fe), "\n")

  nuevas_lag <- unlist(lapply(cols_fe, function(col) paste0(col, c("_lag1", "_delta1"))))
  mostrar_fe(dataset_fe, nuevas_lag, "LAG1 / DELTA1")

  # libero objetos auxiliares
  rm(v, ok, idx_sin_lag, nuevas_lag)
  gc(full = TRUE, verbose = FALSE)

} else cat("Lag1 / Delta1: se omite\n")

Columnas numéricas a aplicar lag1 y delta1: 152 | Columnas totales del dataset: 459 

===== LAG1 / DELTA1 =====
Dimensiones: 983.061 filas x 459 columnas | 2.49 GB
Columnas nuevas/modificadas en este paso: 304 
   numero_de_cliente foto_mes active_quarter_lag1 active_quarter_delta1
               <num>    <int>               <int>                 <int>
1:          12159854   202103                  NA                    NA
2:          12159854   202104                   1                     0
3:          12159854   202105                   1                     0
4:          12159854   202106                   1                     0
5:          12159854   202107                   1                     0
   cliente_vip_lag1 cliente_vip_delta1 internet_lag1 internet_delta1
              <int>              <int>         <int>           <int>
1:               NA                 NA            NA              NA
2:                0                  0             0               0
3:       

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,866850,46.3,1479647,79.1,1479647,79.1
Vcells,335843335,2562.3,525765116,4011.3,364923036,2784.2


#### 03.2 Lag2 y Delta2 (sobre las variables originales)

In [11]:
if (CFG$hacer_fe && isTRUE(CFG$fe$lag2_delta2)) {

  # ---- Feature Engineering: lag2 y delta2 de las variables numericas ORIGINALES ----
  setorder(dataset_fe, numero_de_cliente, foto_mes)

  cols_excluir <- c("numero_de_cliente", "foto_mes", "clase_ternaria")
  cols_orig <- names(dataset_fe)[sapply(dataset_fe, is.numeric)]
  cols_orig <- setdiff(cols_orig[!grepl("_(lag1|delta1|lag2|delta2|rank)$", cols_orig)], cols_excluir)

  periodo0 <- as.integer(dataset_fe$foto_mes / 100) * 12L + dataset_fe$foto_mes %% 100L
  ok <- dataset_fe$numero_de_cliente == shift(dataset_fe$numero_de_cliente, 2L, type = "lag") &
        (periodo0 - shift(periodo0, 2L, type = "lag")) == 2L
  idx_sin_lag2 <- which(is.na(ok) | !ok)
  rm(periodo0, ok)

  for (col in cols_orig) {
    v <- shift(dataset_fe[[col]], 2L, type = "lag")
    v[idx_sin_lag2] <- NA
    set(dataset_fe, j = paste0(col, "_lag2"),   value = v)
    set(dataset_fe, j = paste0(col, "_delta2"), value = dataset_fe[[col]] - v)
  }

  cat("Columnas numericas a aplicar lag2 y delta2:", length(cols_orig), "| Columnas totales del dataset:", ncol(dataset_fe), "\n")
  nuevas_lag2 <- unlist(lapply(cols_orig, function(col) paste0(col, c("_lag2", "_delta2"))))
  mostrar_fe(dataset_fe, nuevas_lag2, "LAG2 / DELTA2")

  # libero objetos auxiliares
  rm(v, idx_sin_lag2, nuevas_lag2, cols_orig)
  gc(full = TRUE, verbose = FALSE)

} else cat("Lag2 / Delta2: se omite\n")

Columnas numericas a aplicar lag2 y delta2: 152 | Columnas totales del dataset: 763 

===== LAG2 / DELTA2 =====
Dimensiones: 983.061 filas x 763 columnas | 4.14 GB
Columnas nuevas/modificadas en este paso: 304 
   numero_de_cliente foto_mes active_quarter_lag2 active_quarter_delta2
               <num>    <int>               <int>                 <int>
1:          12159854   202103                  NA                    NA
2:          12159854   202104                  NA                    NA
3:          12159854   202105                   1                     0
4:          12159854   202106                   1                     0
5:          12159854   202107                   1                     0
   cliente_vip_lag2 cliente_vip_delta2 internet_lag2 internet_delta2
              <int>              <int>         <int>           <int>
1:               NA                 NA            NA              NA
2:               NA                 NA            NA              NA
3:       

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,868297,46.4,1479647,79.1,1479647,79.1
Vcells,557034923,4249.9,757277766,5777.6,615783637,4698.1


#### 03.3 Rank por mes con cero fijo de TODAS las variables
Para cada variable (originales, lags y deltas), por separado dentro de cada mes

In [12]:
if (CFG$hacer_fe && isTRUE(CFG$fe$rank_final)) {

  t0_rank <- Sys.time()
  cols_rank <- setdiff(names(dataset_fe), c("numero_de_cliente", "foto_mes", "clase_ternaria"))
  cols_rank <- cols_rank[vapply(cols_rank, function(cn) is.numeric(dataset_fe[[cn]]), logical(1))]

  rank_mes <- function(x) {
    out <- x
    pos <- !is.na(x) & x > 0
    neg <- !is.na(x) & x < 0
    if (any(pos)) out[pos] <-  frank( x[pos], ties.method = "average") / sum(pos)
    if (any(neg)) out[neg] <- -frank(-x[neg], ties.method = "average") / sum(neg)
    out
  }

  # columna por columna y mes por mes: no duplica el dataset
  idx_mes <- split(seq_len(nrow(dataset_fe)), dataset_fe$foto_mes)
  cat("Rank con cero fijo:", length(cols_rank), "columnas x", length(idx_mes), "meses\n")
  for (k in seq_along(cols_rank)) {
    v <- as.numeric(dataset_fe[[cols_rank[k]]])
    for (ix in idx_mes) v[ix] <- rank_mes(v[ix])
    set(dataset_fe, j = cols_rank[k], value = v)
    if (k %% 100 == 0 || k == length(cols_rank)) {
      cat(sprintf("  [%s] rankeadas %d de %d columnas (%.0f%%) | %.1f min\n", format(Sys.time(), "%H:%M:%S"),
                  k, length(cols_rank), 100 * k / length(cols_rank), as.numeric(difftime(Sys.time(), t0_rank, units = "mins"))))
      flush.console()
      gc(verbose = FALSE)
    }
  }

  cat("Rank final listo:", length(cols_rank), "columnas rankeadas por mes | columnas del dataset:", ncol(dataset_fe), "\n")
  mostrar_fe(dataset_fe, head(cols_rank, 30), "RANK FINAL (todas las variables)")

  rm(cols_rank, idx_mes, v, k, t0_rank)
  gc(full = TRUE, verbose = FALSE)

} else cat("Rank final: se omite\n")

Rank con cero fijo: 760 columnas x 6 meses
  [22:46:45] rankeadas 100 de 760 columnas (13%) | 0.1 min
  [22:46:53] rankeadas 200 de 760 columnas (26%) | 0.2 min
  [22:46:58] rankeadas 300 de 760 columnas (39%) | 0.3 min
  [22:47:05] rankeadas 400 de 760 columnas (53%) | 0.4 min
  [22:47:11] rankeadas 500 de 760 columnas (66%) | 0.5 min
  [22:47:15] rankeadas 600 de 760 columnas (79%) | 0.6 min
  [22:47:20] rankeadas 700 de 760 columnas (92%) | 0.7 min
  [22:47:24] rankeadas 760 de 760 columnas (100%) | 0.8 min
Rank final listo: 760 columnas rankeadas por mes | columnas del dataset: 763 

===== RANK FINAL (todas las variables) =====
Dimensiones: 983.061 filas x 763 columnas | 5.59 GB
Columnas nuevas/modificadas en este paso: 30 
   numero_de_cliente foto_mes active_quarter cliente_vip internet cliente_edad
               <num>    <int>          <num>       <num>    <num>        <num>
1:          12159854   202103      0.5000031           0        0    0.7555157
2:          12159854   20

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,870482,46.5,1479647,79.1,1479647,79.1
Vcells,751277491,5731.8,1090655982,8321.1,1090655390,8321.1


In [13]:
if (CFG$hacer_fe) {
  setorder(dataset_fe, foto_mes, numero_de_cliente)

  write_fst(dataset_fe, CFG$archivo_fe, compress = 50)
  cat("Guardado:", nrow(dataset_fe), "filas x", ncol(dataset_fe), "columnas en", CFG$archivo_fe, "\n")

  # libero: la seccion 04 vuelve a leer solo los meses necesarios (con .fst tarda segundos)
  rm(dataset_fe)
  gc(full = TRUE, verbose = FALSE)
}

Guardado: 983061 filas x 763 columnas en /home/ds/buckets/b1/datasets/competencia_01_fe_naccajas_lag1_delta1_lag2_delta2_rankfinal.fst 


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,868968,46.5,1479647,79.1,1479647,79.1
Vcells,1686006,12.9,872524786,6656.9,1090655390,8321.1


### 04 Optimización de Hiperparámetros

#### 4.1 Inicio

limpio el ambiente de R

In [14]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Fri Oct 09 22:48:24 2026"

In [15]:
# limpio la memoria, conservando solo la configuracion de archivos
rm(list = setdiff(ls(all.names = TRUE), "CFG"))
gc(full = TRUE, verbose = FALSE)

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,868748,46.4,1479647,79.1,1479647,79.1
Vcells,1601791,12.3,698019829,5325.5,1090655390,8321.1


#### 4.2 Carga de Librerias

In [16]:
# cargo las librerias que necesito (se instalan si faltan)
if( !require("data.table") ) install.packages("data.table")
require("data.table")

if( !require("fst") ) install.packages("fst")
require("fst")
require("parallel")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

if( !require("DiceKriging") ) install.packages("DiceKriging")
require("DiceKriging")

if( !require("mlrMBO") ) install.packages("mlrMBO")
require("mlrMBO")

if( !require("ggplot2") ) install.packages("ggplot2")
require("ggplot2")

if( !require("scales") ) install.packages("scales")
require("scales")

if( !require("jsonlite") ) install.packages("jsonlite")
require("jsonlite")


Loading required package: parallel

Loading required package: primes

Loading required package: rlist

Loading required package: yaml

Loading required package: lightgbm

Loading required package: DiceKriging

Loading required package: mlrMBO

Loading required package: mlr

Loading required package: ParamHelpers


Attaching package: ‘ParamHelpers’


The following object is masked from ‘package:R.utils’:

    isVector



Attaching package: ‘mlr’


The following objects are masked from ‘package:R.utils’:

    resample, setThreshold


Loading required package: smoof

Loading required package: checkmate


Attaching package: ‘checkmate’


The following object is masked from ‘package:DiceKriging’:

    checkNames


The following object is masked from ‘package:R.utils’:

    asInt



Attaching package: ‘smoof’


The following objects are masked from ‘package:R.oo’:

    getDescription, getName


Loading required package: ggplot2

Loading required package: scales

Loading required package: jso

#### 4.3 Definición de Hiperparámetros

In [17]:
PARAM <- list()
PARAM$experimento <- "C1_E012"   
PARAM$semilla_primigenia <- 110017

In [18]:
# Esquema: BO entrena 202103-04 y valida en 202105
PARAM$train <- c(202103, 202104)      # entrenamiento de la BO
PARAM$validate <- c(202105)           # validacion de la BO (AUC)

# testing en mes conocido: entreno con train_test y mido ganancia en test
PARAM$train_test <- c(202103, 202104)
PARAM$test <- c(202106)   # ultimo mes con clase_ternaria completa

In [19]:
PARAM$trainingstrategy$undersampling <- 1.0   # SIN undersampling

In [20]:
# Parametros LightGBM
PARAM$hyperparametertuning$xval_folds <- 5        # BO con k-fold 5 estratificado (nfold + stratified)

# parametros fijos del LightGBM (definicion de z495, Gustavo Pasquini)
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", 
  objective= "binary",
  metric= "auc",
  feature_pre_filter= FALSE,
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  force_row_wise= TRUE,
  deterministic= TRUE,
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L,
  min_gain_to_split= 0,
  lambda_l1= 0.0,
  lambda_l2= 0.0,
  max_bin= 31L,

  bagging_fraction= 1.0,
  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  drop_rate= 0.1,
  max_drop= 50,
  skip_drop= 0.5,

  extra_trees= FALSE,

  early_stopping= 0,
  min_data_in_leaf= 0,     # fijo, esto es FUNDAMENTAL
  feature_fraction= 0.50,  # fijo
  learning_rate= 0.005,    # fijo

  num_iterations= 20000,     # se modificara
  num_leaves= 1024,          # se modificara
  min_sum_hessian_in_leaf= 1e-06  # se modificara
)

In [21]:
# Bordes de los hiperparametros de la BO: rangos de z495 (Gustavo Pasquini)
hs_ajustado <- makeParamSet(
  makeIntegerParam("num_iterations", lower= 64L, upper= 4096L),
  makeIntegerParam("num_leaves", lower= 16L, upper= 2048L),
  makeNumericParam("min_sum_hessian_in_leaf", lower= 1e-06, upper= 0.1)  # min_data_in_leaf queda fijo en CERO
)

# BO: k-fold 5 estratificado (nfold + stratified) sobre 202103, 202104 y 202105, SIN undersampling, metrica AUC.
# diseno inicial: el de mlrMBO por defecto (4 x 3 parametros = 12 puntos al azar) + 10 iteraciones bayesianas = 22 evaluaciones
PARAM$bo <- list(
  nombre      = "bo",
  meses       = c(PARAM$train, PARAM$validate),
  iteraciones = 10,
  n_inicial   = 12,   
  hs          = hs_ajustado,
  kbayesiana  = "bayesiana_kfold5_auc.RDATA",   # estado de mlrMBO (para mboContinue)
  log         = "BO_log_kfold5_auc.txt",        # resultado final de la BO (como en el script)
  log_eval    = "BO_eval_kfold5_auc.txt"        # una linea por evaluacion (checkpoint)
)
rm(hs_ajustado)
PARAM$bo_probar_tiempo <- FALSE   # TRUE = correr una evaluacion de prueba (system.time) antes de la BO

# ---- reutilizar los hiperparametros de otro experimento (no se corre la BO) ----
# TRUE : toma el mejor punto de BO_log_kfold5_auc.txt (o BO_eval_kfold5_auc.txt) de la carpeta HT<PARAM$bo_origen>
# FALSE: corre la BO de esta notebook (si ya termino, se toma de su propio log)
PARAM$bo_reutilizar_origen <- FALSE
PARAM$bo_origen <- "C1_E010v2"


In [22]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}

In [23]:
# iniciliazo el dataset de realidad, para medir ganancia
realidad_inicializar <- function( pfuture, pparam) {

  # datos para verificar la ganancia
  drealidad <- pfuture[, list(numero_de_cliente, foto_mes, clase_ternaria)]

  particionar(drealidad,
    division= c(3, 7),
    agrupa= "clase_ternaria",
    seed= PARAM$semilla_kaggle
  )

  return( drealidad )
}

In [24]:
# evaluo ganancia en los datos de la realidad

realidad_evaluar <- function( prealidad, pprediccion) {

  prealidad[ pprediccion,
    on= c("numero_de_cliente", "foto_mes"),
    predicted:= i.Predicted
  ]

  tbl <- prealidad[, list("qty"=.N), list(fold, predicted, clase_ternaria)]

  res <- list()
  res$public  <- tbl[fold==1 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]/0.3
  res$private <- tbl[fold==2 & predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]/0.7
  res$total <- tbl[predicted==1L, sum(qty*ifelse(clase_ternaria=="BAJA+2", 1072500, -27500))]

  prealidad[, predicted:=NULL]
  return( res )
}

#### Funciones de manejo de memoria
Se usan índices de filas en lugar de copiar `dataset` (`dataset[foto_mes %in% ...]` duplica todas las columnas), y las matrices para LightGBM se arman columna por columna.

In [25]:
# filtra filas SIN duplicar el data.table (copia columna por columna y borra la original)
filtrar_filas_bajo_memoria <- function(dt, filas) {
  cols <- copy(names(dt))
  out <- data.table(.fila_tmp = seq_along(filas))
  for (k in seq_along(cols)) {
    set(out, j = cols[k], value = dt[[cols[k]]][filas])
    set(dt, j = cols[k], value = NULL)
    if (k %% 50 == 0) gc(verbose = FALSE)
  }
  out[, .fila_tmp := NULL]
  gc(verbose = FALSE)
  out
}

# matriz para LightGBM sin pasar por un data.table intermedio
# (equivale a data.matrix(dataset[filas, campos, with = FALSE]) pero sin la copia intermedia)
matriz_bajo_memoria <- function(dt, filas, campos) {
  m <- matrix(NA_real_, nrow = length(filas), ncol = length(campos), dimnames = list(NULL, campos))
  for (j in seq_along(campos)) m[, j] <- as.numeric(dt[[campos[j]]][filas])
  m
}

# predict por bloques: nunca arma la matriz completa del mes
predecir_por_bloques <- function(modelo, dt, filas, campos, bloque = 50000L) {
  prob <- numeric(length(filas))
  for (ini in seq(1L, length(filas), by = bloque)) {
    idx <- ini:min(ini + bloque - 1L, length(filas))
    m <- matriz_bajo_memoria(dt, filas[idx], campos)
    prob[idx] <- predict(modelo, m)
    rm(m)
  }
  gc(verbose = FALSE)
  prob
}

# lectura del .fst: solo el rango de filas de los meses pedidos (el archivo esta ordenado por foto_mes)
leer_meses_fst <- function(archivo, meses) {
  fm <- read_fst(archivo, columns = "foto_mes")$foto_mes
  stopifnot(all(meses %in% fm))
  filas <- which(fm %in% meses)
  dt <- read_fst(archivo, from = min(filas), to = max(filas), as.data.table = TRUE)
  rm(fm, filas)
  # meses intermedios que no se usan: se sacan sin duplicar memoria
  filas <- which(dt$foto_mes %in% meses)
  if (length(filas) < nrow(dt)) dt <- filtrar_filas_bajo_memoria(dt, filas)
  gc(full = TRUE, verbose = FALSE)
  dt
}

# ---------- progreso: imprime por pantalla y agrega al archivo estado_<experimento>.txt ----------
# (el archivo queda en la carpeta del experimento; se puede seguir con: tail -f estado_<experimento>.txt)
reportar <- function(...) {
  linea <- paste0("[", format(Sys.time(), "%Y-%m-%d %H:%M:%S"), "] ", paste0(...))
  cat(linea, "\n"); flush.console()
  arch <- file.path(CFG$carpeta_exp, paste0("HT", PARAM$experimento), paste0("estado_", PARAM$experimento, ".txt"))
  if (dir.exists(dirname(arch))) cat(linea, "\n", file = arch, append = TRUE)
  invisible(linea)
}
fmt_min <- function(m) if (is.na(m)) "?" else if (m < 60) sprintf("%.0f min", m) else sprintf("%.1f h", m / 60)


### 4.4  Preprocesamiento

In [27]:
# carpeta base de experimentos (ruta absoluta)
carpeta_exp <- CFG$carpeta_exp
setwd(carpeta_exp)
exp_folder <- paste0("HT", PARAM$experimento)
dir.create(exp_folder, showWarnings = FALSE)
setwd(file.path(carpeta_exp, exp_folder))

[1] "setwd(\"/content/buckets/b1/exp\")\nexperimento_folder <- paste0(\"HT\", PARAM$experimento)\ndir.create(experimento_folder, showWarnings=FALSE)\nsetwd( paste0(\"/content/buckets/b1/exp/\", experimento_folder ))"

In [28]:
# lectura del dataset con FE: solo los meses que se usan en el experimento
meses_usados <- sort(unique(c(PARAM$train, PARAM$train_test, PARAM$test, PARAM$train_final, PARAM$future)))
dataset <- leer_meses_fst(CFG$archivo_fe, meses_usados)

# trazabilidad del preprocesamiento
PARAM$fe <- CFG$fe
PARAM$archivo_fe <- basename(CFG$archivo_fe)

cat("Dataset con FE leido desde:", CFG$archivo_fe, "\n")
cat("Meses:", paste(meses_usados, collapse = ", "), "|", nrow(dataset), "filas x", ncol(dataset), "columnas |",
    round(as.numeric(object.size(dataset)) / 1024^3, 2), "GB\n")

Dataset con FE leido desde: /home/ds/buckets/b1/datasets/competencia_01_fe_naccajas_lag1_delta1_lag2_delta2_rankfinal.fst 
Meses: 202103, 202104, 202105, 202106, 202108 | 818713 filas x 763 columnas | 4.65 GB


In [29]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset),
  c("clase_ternaria", "clase01", "azar", "training")
)

### 4.4b Función objetivo
**BO con k-fold 5 estratificado**, con los rangos de z495, métrica AUC: sobre 202103, 202104 y 202105, **sin undersampling**.
<br>En cada evaluación se agrega una línea a `BO_eval_test_kfold5_auc.txt` con los hiperparámetros, la AUC y el tiempo.

In [30]:
# En el argumento x llegan los parámetros de la bayesiana y devuelve la AUC en cross validation del modelo entrenado

EstimarGanancia_AUC_lightgbm <- function(x) {

  t0_eval <- Sys.time()
  n_total <- PARAM$bo$n_inicial + PARAM$bo$iteraciones
  if (isTRUE(bo_logueando))
    reportar(sprintf("BO | evaluacion %d de ~%d arranca | num_iterations=%d num_leaves=%d min_sum_hessian=%.4g",
                     bo_iter + 1L, n_total, as.integer(x$num_iterations), as.integer(x$num_leaves), x$min_sum_hessian_in_leaf))

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM con k-fold 5 estratificado
  # best_score = AUC promedio en los 5 folds de validacion
  modelocv <- lgb.cv(
    data= dtrain,
    nfold= PARAM$hyperparametertuning$xval_folds,
    stratified= TRUE,
    param= param_completo
  )

  # obtengo la ganancia
  AUC <- modelocv$best_score

  # log por evaluacion (checkpoint: mlrMBO solo graba cada 600 s y entre iteraciones)
  if (isTRUE(bo_logueando)) {
    bo_iter <<- bo_iter + 1L
    fwrite(data.table(fecha = format(Sys.time(), "%Y%m%d %H:%M:%S"), bo = BO$nombre, iter = bo_iter,
                      num_iterations = x$num_iterations, num_leaves = x$num_leaves,
                      min_sum_hessian_in_leaf = x$min_sum_hessian_in_leaf, auc = AUC,
                      minutos = round(as.numeric(difftime(Sys.time(), t0_eval, units = "mins")), 1)),
           BO$log_eval, append = file.exists(BO$log_eval), sep = "\t")

    # progreso: mejor AUC hasta ahora, tiempo promedio por evaluacion y estimacion de lo que falta
    ev <- fread(BO$log_eval)
    min_prom <- mean(ev$minutos)
    faltan <- max(0L, n_total - nrow(ev))
    reportar(sprintf("BO | evaluacion %d de ~%d lista en %s | AUC=%.5f | mejor hasta ahora=%.5f | faltan ~%d (~%s)",
                     nrow(ev), n_total, fmt_min(as.numeric(difftime(Sys.time(), t0_eval, units = "mins"))),
                     AUC, max(ev$auc), faltan, fmt_min(faltan * min_prom)))
  }

  # hago espacio en la memoria
  rm(modelocv)
  gc(full= TRUE, verbose= FALSE)

  message(format(Sys.time(), "%a %b %d %X %Y"), " AUC ", AUC)

  return(AUC)
}

In [31]:
# mejores hiperparametros guardados en una carpeta de experimento (BO_log si termino, si no BO_eval)
mejor_bo_guardada <- function(carpeta, BO) {
  arch_log <- file.path(carpeta, BO$log); arch_eval <- file.path(carpeta, BO$log_eval)
  if (file.exists(arch_log)) {
    tb <- fread(arch_log); origen <- paste(arch_log, "(BO terminada)")
  } else if (file.exists(arch_eval)) {
    tb <- fread(arch_eval); setnames(tb, "auc", "y")
    origen <- paste(arch_eval, "(BO sin terminar:", nrow(tb), "evaluaciones)")
  } else return(NULL)
  setorder(tb, -y)
  list(mejores_hiperparametros = tb[1, list(num_iterations = as.integer(num_iterations),
                                            num_leaves = as.integer(num_leaves),
                                            min_sum_hessian_in_leaf = as.numeric(min_sum_hessian_in_leaf))],
       y = tb[1, y], meses = BO$meses, origen = origen)
}

# carpeta del experimento
dir.create(file.path(CFG$carpeta_exp, paste0("HT", PARAM$experimento)), showWarnings = FALSE)
setwd(file.path(CFG$carpeta_exp, paste0("HT", PARAM$experimento)))

# si la BO ya termino (o se reutiliza la de otro experimento), se toma del log y se saltean las celdas de la BO
res <- if (isTRUE(PARAM$bo_reutilizar_origen)) {
  r <- mejor_bo_guardada(file.path(CFG$carpeta_exp, paste0("HT", PARAM$bo_origen)), PARAM$bo)
  if (is.null(r)) stop("No hay logs de la BO en HT", PARAM$bo_origen)
  r
} else if (file.exists(PARAM$bo$log)) mejor_bo_guardada(getwd(), PARAM$bo) else NULL

if (!is.null(res)) {
  PARAM$out$bo <- res
  cat("BO -> se usa", res$origen, "\n"); print(res$mejores_hiperparametros); cat("AUC:", res$y, "\n")
} else cat("BO -> se corre en esta notebook (o se retoma desde", PARAM$bo$kbayesiana, ")\n")
bo_hecha <- !is.null(res)
rm(res)


BO -> se corre en esta notebook (o se retoma desde bayesiana_kfold5_auc.RDATA )


### 4.5 BO (única): k-fold 5 estratificado sobre 202103–202105, AUC, sin undersampling
Rangos de z495, 12 puntos iniciales + 10 iteraciones bayesianas. Cada evaluación son 5 entrenamientos sobre los tres meses completos, así que es la parte más larga de la notebook: antes y después de cada evaluación se imprime el avance (evaluación N de ~22, AUC, mejor AUC hasta ahora, tiempo estimado restante).
<br>Si se corta, al volver a correr retoma desde `bayesiana_kfold5_auc.RDATA` (mlrMBO graba cada 10 minutos); cada evaluación queda además en `BO_eval_kfold5_auc.txt`.

In [32]:
# BO activa: todas las celdas de este bloque usan BO
BO <- PARAM$bo
bo_logueando <- FALSE
bo_iter <- if (file.exists(BO$log_eval)) nrow(fread(BO$log_eval)) else 0L
cat("BO", BO$nombre, "| meses:", paste(BO$meses, collapse = ", "), "| evaluaciones ya logueadas:", bo_iter, "\n")

BO bo | meses: 202103, 202104, 202105 | evaluaciones ya logueadas: 0 


In [33]:
if (!bo_hecha) {

  # indices de las filas de training de la BO: no se copia el dataset,
  # dataset_train solo lleva las columnas necesarias para la clase, el undersampling y los folds
  filas_bo <- which(dataset$foto_mes %in% BO$meses)
  dataset_train <- dataset[filas_bo, list(numero_de_cliente, foto_mes, clase_ternaria)]

} else cat("BO resuelta: se omite esta celda\n")

In [34]:
if (!bo_hecha) {

  # paso la clase a binaria que tome valores {0,1}  enteros
  #  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0
  #  a partir de ahora ya NO puedo cortar  por prob(BAJA+2) > 1/40

  dataset_train[,
    clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
  ]

} else cat("BO resuelta: se omite esta celda\n")

In [35]:
if (!bo_hecha) {

  # defino los datos que forma parte del training
  # aqui se hace el undersampling de los CONTINUA
  # notar que para esto utilizo la SEGUNDA semilla

  set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
  dataset_train[, azar := runif(nrow(dataset_train))]
  dataset_train[, training := 0L]

  dataset_train[
    foto_mes %in%  BO$meses &
      (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
    training := 1L
  ]

} else cat("BO resuelta: se omite esta celda\n")

In [36]:
if (!bo_hecha) {

  # dejo los datos en el formato que necesita LightGBM

  dtrain <- lgb.Dataset(
    data= matriz_bajo_memoria(dataset, filas_bo[dataset_train$training == 1L], campos_buenos),
    label= dataset_train[training == 1L, clase01],
    params= list(max_bin = PARAM$lgbm$param_fijos$max_bin, feature_pre_filter = FALSE, seed = PARAM$semilla_primigenia),
    free_raw_data= TRUE
  )
  lgb.Dataset.construct(dtrain)
  cat("BO", BO$nombre, "| k-fold", PARAM$hyperparametertuning$xval_folds, "estratificado sobre",
      paste(BO$meses, collapse = ", "), "(", nrow(dtrain), "filas con undersampling", PARAM$trainingstrategy$undersampling, ")\n")
  gc(full = TRUE, verbose = FALSE)

  nrow(dtrain)
  ncol(dtrain)

} else cat("BO resuelta: se omite esta celda\n")

BO bo | k-fold 5 estratificado sobre 202103, 202104, 202105 ( 489952 filas con undersampling 1 )


[1] 762

In [37]:
if (!bo_hecha) {

  # Folds agrupados por cliente

  armar_folds_agrupados <- function(dt, k, seed) {
    set.seed(seed, kind = "L'Ecuyer-CMRG")
    cli <- dt[, list(pos = max(clase01)), by = numero_de_cliente]
    cli[, fold := sample(rep(seq_len(k), length.out = .N)), by = pos]
    fold_fila <- cli[dt[, list(numero_de_cliente)], on = "numero_de_cliente", fold]
    lapply(seq_len(k), function(i) which(fold_fila == i))
  }

  folds_grupales <- armar_folds_agrupados(
    dataset_train[training == 1L],
    k = PARAM$hyperparametertuning$xval_folds,
    seed = PARAM$semilla_primigenia
  )

  # dtrain ya tiene los datos (free_raw_data = FALSE): libero la tabla de training
  rm(dataset_train, filas_bo)
  gc(full = TRUE, verbose = FALSE)

} else cat("BO resuelta: se omite esta celda\n")

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2474151,132.2,4902564,261.9,4902564,261.9
Vcells,629344243,4801.6,1350914744,10306.7,1220735837,9313.5


In [38]:
if (!bo_hecha) {

  # Aqui comienza la configuracion de la Bayesian Optimization

  # en este archivo quedan la evolucion binaria de la BO
  kbayesiana <- BO$kbayesiana

  funcion_optimizar <- EstimarGanancia_AUC_lightgbm # la funcion que voy a maximizar

  configureMlr(show.learner.output= FALSE)

  # configuro la busqueda bayesiana,  los hiperparametros que se van a optimizar
  # por favor, no desesperarse por lo complejo

  obj.fun <- makeSingleObjectiveFunction(
    fn= funcion_optimizar, # la funcion que voy a maximizar
    minimize= FALSE, # estoy Maximizando la ganancia
    noisy= TRUE,
    par.set= BO$hs, # definido al comienzo del programa
    has.simple.signature= FALSE # paso los parametros en una lista
  )

  # cada 600 segundos guardo el resultado intermedio
  ctrl <- makeMBOControl(
    save.on.disk.at.time= 600, # se graba cada 600 segundos
    save.file.path= kbayesiana
  ) # se graba cada 600 segundos

  # indico la cantidad de iteraciones que va a tener la Bayesian Optimization
  ctrl <- setMBOControlTermination(
    ctrl,
    iters= BO$iteraciones
  ) # cantidad de iteraciones

  # defino el método estandar para la creacion de los puntos iniciales,
  # los "No Inteligentes"
  ctrl <- setMBOControlInfill(ctrl, crit= makeMBOInfillCritEI())

  # establezco la funcion que busca el maximo
  surr.km <- makeLearner(
    "regr.km",
    predict.type= "se",
    covtype= "matern3_2",
    control= list(trace= TRUE)
  )

} else cat("BO resuelta: se omite esta celda\n")

In [40]:
if (!bo_hecha) {

  # inicio la optimizacion bayesiana, retomando si ya existe
  # es la celda mas lenta de todo el notebook

  bo_logueando <- TRUE   # desde aca cada evaluacion se loguea en BO$log_eval

  reportar("BO | arranca: ", PARAM$bo$n_inicial, " puntos iniciales + ", BO$iteraciones, " iteraciones bayesianas | ",
           "evaluaciones ya hechas: ", bo_iter)
  if (!file.exists(kbayesiana)) {
    bayesiana_salida <- mbo(obj.fun, learner= surr.km, control= ctrl)
  } else {
    print("BO: retomo corrida ...")
    bayesiana_salida <- mboContinue(kbayesiana) # retomo en caso que ya exista
  }

  bo_logueando <- FALSE
  reportar("BO | terminada")

} else cat("BO resuelta: se omite esta celda\n")

[2026-10-09 22:49:49] BO | arranca: 12 puntos iniciales + 10 iteraciones bayesianas | evaluaciones ya hechas: 0 


Computing y column(s) for design. Not provided.



[2026-10-09 22:49:50] BO | evaluacion 1 de ~22 arranca | num_iterations=1377 num_leaves=1217 min_sum_hessian=0.08687 
[2026-10-09 23:39:32] BO | evaluacion 1 de ~22 lista en 50 min | AUC=0.94107 | mejor hasta ahora=0.94107 | faltan ~21 (~17.4 h) 


Fri Oct 09 23:39:33 2026 AUC 0.941068707404153



[2026-10-09 23:39:33] BO | evaluacion 2 de ~22 arranca | num_iterations=1821 num_leaves=776 min_sum_hessian=0.04295 
[2026-10-10 00:32:09] BO | evaluacion 2 de ~22 lista en 53 min | AUC=0.94107 | mejor hasta ahora=0.94107 | faltan ~20 (~17.1 h) 


...

[2026-10-10 14:08:10] BO | evaluacion 15 de ~22 arranca | num_iterations=2919 num_leaves=1619 min_sum_hessian=0.09999 
[2026-10-10 15:45:22] BO | evaluacion 15 de ~22 lista en 1.6 h | AUC=0.94177 | mejor hasta ahora=0.94342 | faltan ~7 (~7.9 h) 


Sat Oct 10 15:45:23 2026 AUC 0.941774628551763

[mbo] 3: num_iterations=2919; num_leaves=1619; min_sum_hessian_in_leaf=0.1 : y = 0.942 : 5832.9 secs : infill_ei

Saved the current state after iteration 4 in the file bayesiana_kfold5_auc.RDATA.



[2026-10-10 15:45:24] BO | evaluacion 16 de ~22 arranca | num_iterations=3102 num_leaves=1089 min_sum_hessian=0.07009 
[2026-10-10 17:23:38] BO | evaluacion 16 de ~22 lista en 1.6 h | AUC=0.94259 | mejor hasta ahora=0.94342 | faltan ~6 (~7.0 h) 


Sat Oct 10 17:23:39 2026 AUC 0.942593642203026

[mbo] 4: num_iterations=3102; num_leaves=1089; min_sum_hessian_in_leaf=0.0701 : y = 0.943 : 5895.1 secs : infill_ei

Saved the current state after iteration 5 in the file bayesiana_kfold5_auc.RDATA.



[2026-10-10 17:23:40] BO | evaluacion 17 de ~22 arranca | num_iterations=2460 num_leaves=1387 min_sum_hessian=1.603e-05 


In [41]:
# BO cortada a mano a las 16 evaluaciones inclusive: arma los archivos finales con las evaluaciones hechas
setwd(file.path(CFG$carpeta_exp, paste0("HT", PARAM$experimento)))

ev <- fread(PARAM$bo$log_eval)   # una fila por evaluacion terminada
tb_bayesiana <- tryCatch(as.data.table(mboFinalize(PARAM$bo$kbayesiana)$opt.path),
                         error = function(e) NULL)

# si el estado de mlrMBO tiene menos evaluaciones que BO_eval (se graba cada 10 min), se arma desde BO_eval
if (is.null(tb_bayesiana) || nrow(tb_bayesiana) < nrow(ev)) {
  tb_bayesiana <- ev[, list(num_iterations, num_leaves, min_sum_hessian_in_leaf, y = auc)]
  cat("BO_log armado desde", PARAM$bo$log_eval, "\n")
} else cat("BO_log armado desde", PARAM$bo$kbayesiana, "(mboFinalize)\n")

tb_bayesiana[, iter := .I]
setorder(tb_bayesiana, -y)
fwrite(tb_bayesiana, file = PARAM$bo$log, sep = "\t")

PARAM$out$bo$mejores_hiperparametros <- tb_bayesiana[1, list(num_iterations = as.integer(num_iterations),
                                                            num_leaves = as.integer(num_leaves),
                                                            min_sum_hessian_in_leaf = as.numeric(min_sum_hessian_in_leaf))]
PARAM$out$bo$y <- tb_bayesiana[1, y]
PARAM$out$bo$meses <- PARAM$bo$meses
PARAM$out$bo$evaluaciones <- nrow(tb_bayesiana)
write_yaml(PARAM, file = "PARAM.yml")

bo_hecha <- TRUE   # las celdas siguientes de la BO se saltean
cat("BO cortada en", nrow(tb_bayesiana), "evaluaciones\n")
print(PARAM$out$bo$mejores_hiperparametros, digits = 17); cat("AUC:", PARAM$out$bo$y, "\n")

Saved the current state after iteration 5 in the file bayesiana_kfold5_auc.RDATA.



BO_log armado desde bayesiana_kfold5_auc.RDATA (mboFinalize)
BO cortada en 16 evaluaciones
   num_iterations num_leaves min_sum_hessian_in_leaf
            <int>      <int>                   <num>
1:           2891       1056    0.013485676441085981
AUC: 0.9434219 
